In [2]:
!pip install yfinance
!pip install altair
!pip install scipy
!pip install pandas
!pip install numpy
!pip install matplotlib

In [25]:
import yfinance as yf
import pandas as pd
import numpy as np
import altair as alt
from scipy.optimize import newton

alt.renderers.enable("html")
AED_USD = 1/3.67

In [26]:
property_val = 510_000 * AED_USD
loan_amount = property_val * 0.8
emi = 3_500 * AED_USD
n_period = 12*12

def pv(r, n, pmt, fv=0, beg=False):
    c = np.full(n, pmt)
    t = np.arange(1, n+1)
    d = (1. / np.power((1 + r), t))
    B = np.sum(d * c)
    tv = fv / (1 + r)**n

    return np.where(beg, (B + tv) * (1 + r), B + tv)

def f(r):
    return pv(r=r, n=n_period, pmt=emi, beg=False) - loan_amount

interest_rate, status = newton(f, 0, full_output=True, disp=False)

amort_schedule = pd.DataFrame()
amort_schedule['Month'] = np.arange(1, n_period+1)
amort_schedule['Beg Balance'] = (
    loan_amount *
    ((1 + interest_rate)**n_period - (1 + interest_rate)**(amort_schedule['Month']-1)) /
    ((1 + interest_rate)**n_period - 1)
)
amort_schedule['End Balance'] = (
    loan_amount *
    ((1 + interest_rate)**n_period - (1 + interest_rate)**(amort_schedule['Month'])) /
    ((1 + interest_rate)**n_period - 1)
)
amort_schedule['Interest'] = amort_schedule['Beg Balance'] * interest_rate
amort_schedule['Principal'] = emi - amort_schedule['Interest']
amort_schedule['EMI'] = emi

amort_schedule

,Month,Beg Balance,End Balance,Interest,Principal,EMI
0,1,111171.662125,110554.604621,336.620970,617.057505,953.678474
1,2,110554.604621,109935.678704,334.752557,618.925917,953.678474
2,3,109935.678704,109314.878717,332.878488,620.799987,953.678474
3,4,109314.878717,108692.198987,330.998743,622.679731,953.678474
4,5,108692.198987,108067.633820,329.113307,624.565167,953.678474
...,...,...,...,...,...,...
139,140,4725.381359,3786.011050,14.308165,939.370309,953.678474
140,141,3786.011050,2843.796385,11.463809,942.214665,953.678474
141,142,2843.796385,1898.728752,8.610841,945.067633,953.678474
142,143,1898.728752,950.799512,5.749234,947.929240,953.678474


In [48]:
start_date = pd.to_datetime("2020-01-01")
end_date = pd.DateOffset(months=145) + start_date

annual_rent = 36_000
annual_rent_gwth = 0.05
annual_appreciation = 0.05

# Home equity and income schedule
home_equity = pd.DataFrame()
home_equity['Date'] = pd.date_range(start=start_date, end=end_date, freq="MS")
home_equity['Home Value'] = property_val * (1 + annual_appreciation/12)**np.arange(n_period+2)
home_equity

,Date,Home Value
0,2020-01-01,138964.577657
1,2020-02-01,139543.596730
2,2020-03-01,140125.028383
3,2020-04-01,140708.882668
4,2020-05-01,141295.169679
...,...,...
141,2031-10-01,249759.509933
142,2031-11-01,250800.174558
143,2031-12-01,251845.175285
144,2032-01-01,252894.530182


In [39]:
property_val

138964.57765667577

In [6]:
# Vanguard S&P 500 ETF (VOO)
snp500 = yf.Ticker("VOO")
start_date = "2020-01-01"
end_date = "2025-06-16"
snp500_hist = snp500.history(start=start_date, end=end_date)
snp500_hist.reset_index(inplace=True)

In [7]:
alt.Chart(snp500_hist).mark_line().encode(
    x="Date:T",
    y="Close:Q"
).properties(
    width='container',
    height=300
)

alt.Chart(...)

In [8]:
# Investment Portfolio
monthly_investment = (np.where(np.diff(snp500_hist['Date'].dt.month, prepend=1) > 0, emi, 0))
monthly_investment[0] = 130_000 * AED_USD

investment_df = pd.DataFrame()
investment_df['Date'] = snp500_hist['Date']
investment_df['Investments'] = monthly_investment
investment_df['Delta Shares'] = investment_df['Investments'] / snp500_hist['Open']
investment_df['Total Shares'] = investment_df['Delta Shares'].cumsum()
investment_df['Portfolio Value'] = investment_df['Total Shares'] * snp500_hist['Close']
investment_df['Avg Price'] = investment_df['Investments'].cumsum() / investment_df['Total Shares']
investment_df['Dividend Income'] = investment_df['Total Shares'] * snp500_hist['Dividends']

alt.Chart(investment_df).mark_line().encode(
    x="Date:T",
    y="Portfolio Value:Q"
).properties(
    width='container',
    height=300
)

alt.Chart(...)

In [11]:
print(f"Total Investment (USD) = {investment_df['Investments'].sum():,.2f}")
print(f"Total Investment (AED) = {investment_df['Investments'].sum() / AED_USD:,.2f}")

print(f"Portfolio Value (USD) = {investment_df['Portfolio Value'].to_numpy()[-1]:,.2f}")
print(f"Portfolio Value (AED) = {investment_df['Portfolio Value'].to_numpy()[-1] / AED_USD:,.2f}")

print(f"Dividend Income = {investment_df['Dividend Income'].sum():,.2f}")
print(f"Dividend Income = {investment_df['Dividend Income'].sum() / AED_USD:,.2f}")

Total Investment (USD) = 92,643.05
Total Investment (AED) = 340,000.00
Portfolio Value (USD) = 155,937.69
Portfolio Value (AED) = 572,291.33
Dividend Income = 6,945.64
Dividend Income = 25,490.48
